In [1]:
%pip install chromadb langchain pypdf2 tiktoken streamlit python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

import streamlit as st

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains.question_answering import load_qa_chain
from langchain.chat_models import ChatOpenAI
from langchain.vectorstores import Chroma
import chromadb

e:\IDEs-tools\anaconda\envs\rag-env\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in HuggingFaceInferenceAPIEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [14]:
def load_chunk_persist_pdf() -> Chroma:
    pdf_folder_path = "./archive/data-trimmed/ALL"
    documents = []
    for file in os.listdir(pdf_folder_path):
        if file.endswith('.pdf'):
            pdf_path = os.path.join(pdf_folder_path, file)
            loader = PyPDFLoader(pdf_path)
            documents.extend(loader.load())
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=10)
    chunked_documents = text_splitter.split_documents(documents)
    client = chromadb.Client()
    if client.list_collections():
        resume_collection = client.create_collection("resume_collection")
    embeddingsFunction = OpenAIEmbeddings(model="text-embedding-ada-002", chunk_size=1, openai_api_key=OPEN_API_KEY)
    vectordb = Chroma.from_documents(
        documents=chunked_documents,
        embedding=embeddingsFunction,
        persist_directory="E:\\SynergyTech\\chat-assistant\\ResumeScan-Agent\\vector_db\\"
    )
    vectordb.persist()
    return vectordb

In [15]:
def create_agent_chain():
    model_name = "gpt-3.5-turbo"
    llm = ChatOpenAI(model_name=model_name, openai_api_key=OPEN_API_KEY)
    chain = load_qa_chain(llm, chain_type="stuff")
    return chain

In [16]:
def get_chain_matching_docs(query):
    vectordb = load_chunk_persist_pdf()
    chain = create_agent_chain()
    matching_docs = vectordb.similarity_search(query)
    #answer = chain.run(input_documents=matching_docs, question=query)
    return [matching_docs, chain]

In [17]:
def get_llm_response(chain, matching_docs, query):
    return chain.run(input_documents=matching_docs, question=query)

In [29]:
query = 'What is the maximum amount of experience in Aviation?'
result = get_chain_matching_docs(query)
answer = get_llm_response(result[1], result[0], query)
print(f"Question: {query}")
print(answer)

Question: What is the maximum amount of experience in Aviation?
The maximum amount of experience in Aviation listed in the provided context is from January 2003 to the present day (Current position).


In [30]:
query = 'Which skills are mentioned in the Accounting Resumes?'
result = get_chain_matching_docs(query)
answer = get_llm_response(result[1], result[0], query)
print(f"Question: {query}")
print(answer)

Question: Which skills are mentioned in the Accounting Resumes?
The skills mentioned in the accounting resume are:
- Intermediate Word
- Advanced Excel
- PowerPoint Intermediate
- Access
- Accounts Receivable
- Accounts Payable
- QuickBooks Enterprise
- Outlook
- Customer Service


In [31]:
query = 'What expertise are seen within the candidates applying for Agriculture jobs?'
result = get_chain_matching_docs(query)
answer = get_llm_response(result[1], result[0], query)
print(f"Question: {query}")
print(answer)

Question: What expertise are seen within the candidates applying for Agriculture jobs?
Based on the provided context, candidates applying for agriculture jobs, particularly in plant protection and quarantine, have expertise in data collection, forest and insect field data analysis, plant identification, invasive species management (such as Asian Long-Horned Beetle and Emerald Ash Borer), ground-based visual surveys, forest insect pest monitoring, and forest canopy protection. They are also skilled in operating equipment like insect traps, 4x4 pickup trucks, and computers with mapping software. Additionally, they possess knowledge about plant hosts, forest ecosystems, and environmental conservation practices.
